# Record only the fact of a connection

This record says that one product belongs to one category. Its identifier and creation timestamp are service data. There is no price, quantity, or agreement to describe.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [2]:
from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationOne, BaseAssocEntity, BaseCoreEntity, NoInverse
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage

## Declare the model

In [3]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Product", domain=ShopDomain)
class ProductCoreEntity(BaseCoreEntity):
    """Identify a product in the catalogue."""

    id: str = Field(description="Product identifier")


@entity(description="Catalogue category", domain=ShopDomain)
class CategoryCoreEntity(BaseCoreEntity):
    """Identify a category used to organise products."""

    id: str = Field(description="Category identifier")


@entity(description="Product membership in a category", domain=ShopDomain)
class ProductCategoryAssocEntity(BaseAssocEntity):
    """Record a technical connection between product and category."""

    id: str = Field(description="Storage identifier of the link")
    created_at: str = Field(description="Storage insertion timestamp")
    product: Annotated[AssociationOne[ProductCoreEntity], NoInverse()] = Field(description="Linked product")
    category: Annotated[AssociationOne[CategoryCoreEntity], NoInverse()] = Field(description="Linked category")


@meta(description="Stores the shop entities")
@entities(
    ProductCoreEntity,
    CategoryCoreEntity,
    ProductCategoryAssocEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""

## Run this case

In [4]:
link = ProductCategoryAssocEntity(
    id="link-1", created_at="2026-10-22T09:00:00Z",
    product=AssociationOne[ProductCoreEntity](id="product-1"),
    category=AssociationOne[CategoryCoreEntity](id="category-1"),
)
assert link.product.id == "product-1" and link.category.id == "category-1"
print(link.product.id, "belongs to", link.category.id)

product-1 belongs to category-1


AssociationOne holds a reference, here only an identifier. NoInverse declares that this example has no reverse field. The Assoc marker describes the whole link record. An ordinary Core entity can also have AssociationOne fields; a relationship field alone does not make its owner an Assoc entity.